In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기
# ============================================================

train = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '2_train_clean.parquet')
val = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '1_val_fe.parquet')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
# ============================================================
# 1. 라이브러리 불러오기
# ============================================================

import numpy as np
import optuna

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

c:\Users\20162180\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# =============================================================
# 2. train 데이터를 튜닝 검증용으로 재분할(val은 최종 val로 남겨둠)
# =============================================================
# 2-1. train -> train_optuna, val_optuna

# 기존 train의 event_id 확인
print(f"train의 event_id 개수: {train['event_id'].nunique()}")  # 98개

optuna_split_idx = int((train['event_id'].nunique()) * 0.8)
print(f"재분할 train_optuna event_id 개수: {optuna_split_idx}")
print()

train_optuna = train.loc[
    train['event_id'] <= optuna_split_idx
]
val_optuna = train.loc[
    train['event_id'] > optuna_split_idx
]

print(f"전체 train event_id 개수: {train['event_id'].nunique()}")
print(f"train_optuna event_id 개수: {train_optuna['event_id'].nunique()}")
print(f"val_optuna event_id 개수: {val_optuna['event_id'].nunique()}")

train의 event_id 개수: 98
재분할 train_optuna event_id 개수: 78

전체 train event_id 개수: 98
train_optuna event_id 개수: 78
val_optuna event_id 개수: 20


In [5]:
# =============================================================
# 3. 종속변수 설정
# =============================================================

y_train_optuna = train_optuna[target_col]
y_val_optuna = val_optuna[target_col]

In [6]:
print(len(train_optuna['station_id'].unique()))
print(len(val_optuna['station_id'].unique()))
print(
    set(train_optuna['station_id'].unique()) - 
    set(val_optuna['station_id'].unique())
    ) # set() 비어있으면 둘은 같은 집합, OHE 문제 없음

465
465
set()


In [7]:
# ============================================================
# 4. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_optuna = train_optuna[num_cols_fe]
X_val_optuna = val_optuna[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_optuna[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_optuna[['station_id']])
val_st_id_ohe = ohe.transform(val_optuna[['station_id']])
# 여기까지는 일반적인 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_optuna_ohe = hstack([
    X_train_optuna,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_val_optuna_ohe = hstack([
    X_val_optuna,
    val_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남")
print(X_train_optuna.shape, X_train_optuna_ohe.shape)
print(X_val_optuna.shape, X_val_optuna_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 늘어남
(404009, 31) (404009, 496)
(212309, 31) (212309, 496)


In [15]:
# ============================================================
# 5. optuna objective함수 정의
# ============================================================

# 라이브러리 호출은 위에 해놨음

def objective(trial):

    # 1. 하이퍼파라미터 탐색
    depth = trial.suggest_int(
        'depth',
        4,
        9
    )

    learning_rate = trial.suggest_float(
        'learning_rate',
        0.01,
        0.1,
        log=True
    )

    l2_leaf_reg = trial.suggest_float(
        'l2_leaf_reg',
        1.0,
        10.0,
        log=True
    )

    random_strength = trial.suggest_float(
        'random_strength',
        0.01,
        10.0,
        log=True
    )

    # 2. 모델 정의
    model = CatBoostRegressor(
        iterations=800,  # 최대 800회 부스팅까지 허용
        depth=depth,
        learning_rate=learning_rate,
        l2_leaf_reg=l2_leaf_reg,
        random_strength=random_strength,
        loss_function='RMSE',
        random_seed=42,
        thread_count=2,  # RF의 n_jobs와 동일: 병렬작업 사용개수
        verbose=100,  # 100개 트리마다 현황정보 표기
        allow_writing_files=False  # 학습 중 로그 등을 별도로 기록, 분석과 무관하므로 불필요
    )

    # 3. 모델 학습
    model.fit(
        X_train_optuna_ohe,
        y_train_optuna,
        eval_set=(X_val_optuna_ohe, y_val_optuna),  # 내부 검증셋
        early_stopping_rounds=50  # RMSE가 50번 연속 개선되지 않으면 종료
    )

    # 4. 검증 데이터 예측
    pred = model.predict(
        X_val_optuna_ohe
    )

    # 5. RMSE 계산
    rmse = root_mean_squared_error(
        y_val_optuna,
        pred
    )

    # 6. Optuna에 RMSE 반환
    return rmse

In [16]:
# ============================================================
# 6. Optuna Study 생성 및 실행
# ============================================================

study = optuna.create_study(
    direction='minimize',
    sampler=optuna.samplers.TPESampler(
        seed=42
    )
)

study.optimize(
    objective,
    n_trials=3
)

[I 2026-10-19 17:19:04,798] A new study created in memory with name: no-name-d2c59496-fc3a-470c-bfdb-57d39607d584


0:	learn: 0.3599489	test: 0.4329118	best: 0.4329118 (0)	total: 74.3ms	remaining: 59.4s
100:	learn: 0.1928834	test: 0.3186160	best: 0.3186160 (100)	total: 6.82s	remaining: 47.2s
200:	learn: 0.1813379	test: 0.3141314	best: 0.3140170 (191)	total: 13.5s	remaining: 40.2s
300:	learn: 0.1745605	test: 0.3119989	best: 0.3119989 (300)	total: 20.2s	remaining: 33.5s
400:	learn: 0.1696884	test: 0.3109065	best: 0.3108905 (398)	total: 26.7s	remaining: 26.6s
500:	learn: 0.1655184	test: 0.3101546	best: 0.3101253 (496)	total: 33.3s	remaining: 19.9s
600:	learn: 0.1621544	test: 0.3096515	best: 0.3096385 (591)	total: 39.9s	remaining: 13.2s
700:	learn: 0.1591109	test: 0.3095313	best: 0.3095190 (673)	total: 46.5s	remaining: 6.56s
799:	learn: 0.1564526	test: 0.3093240	best: 0.3093240 (799)	total: 53.2s	remaining: 0us

bestTest = 0.3093239543
bestIteration = 799



[I 2026-10-19 17:20:00,588] Trial 0 finished with value: 0.30932395456510775 and parameters: {'depth': 6, 'learning_rate': 0.08927180304353628, 'l2_leaf_reg': 5.395030966670228, 'random_strength': 0.6251373574521749}. Best is trial 0 with value: 0.30932395456510775.


0:	learn: 0.3752388	test: 0.4451225	best: 0.4451225 (0)	total: 54.7ms	remaining: 43.7s
100:	learn: 0.2542515	test: 0.3537901	best: 0.3537901 (100)	total: 4.9s	remaining: 33.9s
200:	learn: 0.2304958	test: 0.3394205	best: 0.3394205 (200)	total: 9.82s	remaining: 29.3s
300:	learn: 0.2199382	test: 0.3329130	best: 0.3329130 (300)	total: 14.8s	remaining: 24.6s
400:	learn: 0.2134582	test: 0.3289417	best: 0.3289417 (400)	total: 20s	remaining: 19.9s
500:	learn: 0.2090644	test: 0.3258803	best: 0.3258803 (500)	total: 25.1s	remaining: 15s
600:	learn: 0.2058205	test: 0.3234778	best: 0.3234778 (600)	total: 30.3s	remaining: 10s
700:	learn: 0.2032851	test: 0.3217128	best: 0.3217128 (700)	total: 35.7s	remaining: 5.04s
799:	learn: 0.2012863	test: 0.3204823	best: 0.3204823 (799)	total: 40.7s	remaining: 0us

bestTest = 0.3204822806
bestIteration = 799



[I 2026-10-19 17:20:43,579] Trial 1 finished with value: 0.3204822808618215 and parameters: {'depth': 4, 'learning_rate': 0.01432169828911152, 'l2_leaf_reg': 1.1430983876313219, 'random_strength': 3.9676050770529883}. Best is trial 0 with value: 0.30932395456510775.


0:	learn: 0.3681010	test: 0.4394793	best: 0.4394793 (0)	total: 89.4ms	remaining: 1m 11s
100:	learn: 0.2027804	test: 0.3233159	best: 0.3233159 (100)	total: 8.48s	remaining: 58.7s
200:	learn: 0.1910971	test: 0.3174701	best: 0.3174701 (200)	total: 18.7s	remaining: 55.8s
300:	learn: 0.1830463	test: 0.3146838	best: 0.3146672 (298)	total: 30.3s	remaining: 50.2s
400:	learn: 0.1762266	test: 0.3124704	best: 0.3124704 (400)	total: 39.6s	remaining: 39.4s
500:	learn: 0.1708968	test: 0.3112224	best: 0.3112224 (500)	total: 49.8s	remaining: 29.7s
600:	learn: 0.1666130	test: 0.3104160	best: 0.3104160 (600)	total: 58.4s	remaining: 19.3s
700:	learn: 0.1628825	test: 0.3099303	best: 0.3098916 (688)	total: 1m 7s	remaining: 9.52s
799:	learn: 0.1597143	test: 0.3094163	best: 0.3094163 (799)	total: 1m 15s	remaining: 0us

bestTest = 0.309416343
bestIteration = 799



[I 2026-10-19 17:22:01,638] Trial 2 finished with value: 0.3094163432578105 and parameters: {'depth': 7, 'learning_rate': 0.051059032093947576, 'l2_leaf_reg': 1.0485387725194617, 'random_strength': 8.123245085588687}. Best is trial 0 with value: 0.30932395456510775.


In [17]:
y_val = val[target_col]

for name, y in [
    ('Optuna Train', y_train_optuna),
    ('Optuna Val', y_val_optuna),
    ('기존 Val', y_val)
]:
    print(f'\n{name}')
    print(y.describe(
        percentiles=[0.5, 0.9, 0.95, 0.99]
    ))


Optuna Train
count    404009.000000
mean          0.099586
std           0.377875
min          -2.180000
50%           0.010000
90%           0.230000
95%           0.490000
99%           1.760000
max           7.980000
Name: target_maxrise_6h, dtype: float64

Optuna Val
count    212309.000000
mean          0.134761
std           0.445855
min          -2.070000
50%           0.010000
90%           0.380000
95%           0.780000
99%           2.190000
max           8.300000
Name: target_maxrise_6h, dtype: float64

기존 Val
count    133325.000000
mean          0.096447
std           0.357300
min          -2.120000
50%           0.010000
90%           0.240000
95%           0.510000
99%           1.560000
max           7.710000
Name: target_maxrise_6h, dtype: float64


In [18]:
# 기존에 썼던 기본 CatBoost로 RMSE 평가
from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

model_optuna_cat = CatBoostRegressor(
    random_state = 42)

# 학습
model_optuna_cat.fit(X_train_optuna_ohe, y_train_optuna)

# 예측
pred_optuna_cat = model_optuna_cat.predict(X_val_optuna_ohe)

# 검증
rmse_optuna_cat = root_mean_squared_error(
    y_val_optuna,
    pred_optuna_cat
)

print(rmse_optuna_cat)

Learning rate set to 0.105679
0:	learn: 0.3564758	total: 53.8ms	remaining: 53.7s
1:	learn: 0.3383244	total: 115ms	remaining: 57.6s
2:	learn: 0.3225444	total: 169ms	remaining: 56.1s
3:	learn: 0.3085940	total: 210ms	remaining: 52.3s
4:	learn: 0.2966131	total: 253ms	remaining: 50.4s
5:	learn: 0.2864217	total: 299ms	remaining: 49.5s
6:	learn: 0.2774220	total: 370ms	remaining: 52.4s
7:	learn: 0.2697453	total: 423ms	remaining: 52.4s
8:	learn: 0.2629555	total: 470ms	remaining: 51.7s
9:	learn: 0.2569664	total: 513ms	remaining: 50.8s
10:	learn: 0.2522102	total: 557ms	remaining: 50.1s
11:	learn: 0.2476619	total: 605ms	remaining: 49.8s
12:	learn: 0.2436037	total: 649ms	remaining: 49.3s
13:	learn: 0.2400954	total: 697ms	remaining: 49.1s
14:	learn: 0.2370477	total: 755ms	remaining: 49.6s
15:	learn: 0.2344863	total: 812ms	remaining: 49.9s
16:	learn: 0.2321657	total: 872ms	remaining: 50.4s
17:	learn: 0.2301516	total: 925ms	remaining: 50.5s
18:	learn: 0.2282886	total: 975ms	remaining: 50.3s
19:	learn: